In [1]:
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from torchvision.models import resnet18,ResNet18_Weights
from torchvision import transforms
from modules import creation_dataframe,CreationDataset,Train,DistilbertResnetModel

In [2]:
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
train_dataset=CreationDataset(train_df)
val_dataset=CreationDataset(val_df)

In [4]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
def collate_fn(batch):
    images=[b["images"] for b in batch]
    texts=[b["texts"] for b in batch]
    labels=[b["labels"] for b in batch]
    texts=tokenizer(texts,return_tensors="pt",max_length=128,padding="max_length",truncation=True)
    images=torch.stack(images,dim=0)
    labels=torch.tensor(labels,dtype=torch.long)
    inputs={"images":images,"labels":labels}
    inputs.update(texts)
    return inputs

In [6]:
batch_size=32
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_fn)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_fn)

In [7]:
resnet_model=resnet18(weights=ResNet18_Weights.DEFAULT)

In [8]:
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")

In [9]:
model=DistilbertResnetModel(distilbert_model,resnet_model)

In [10]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [11]:
head_paramas=[model.projection_image.parameters(),model.fc_layers.parameters(),model.fc_norm_layers.parameters(),model.layer_norm_mha.parameters(),model.mha.parameters()]
backbone_parameters=[model.distilbert_model.parameters(),model.resnet_model.parameters()]
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=0.1*n_steps
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [12]:
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4)

In [13]:
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings")

2026-02-22 20:08:07.703 | INFO     | modules.Train:run_training:88 - Epoch 0 :
2026-02-22 20:43:00.527 | INFO     | modules.Train:run_training:141 - Epoch 0: Train Loss = 0.7146985927470645
2026-02-22 20:43:00.529 | INFO     | modules.Train:run_training:142 - Epoch 0: Train Accuracy = 0.5261176470588236
2026-02-22 20:43:00.530 | INFO     | modules.Train:run_training:143 - Epoch 0: Train F1 = 0.5302187225171999
2026-02-22 20:43:00.531 | INFO     | modules.Train:run_training:145 - Epoch 0: Validation Loss = 0.7248550914227962
2026-02-22 20:43:00.534 | INFO     | modules.Train:run_training:146 - Epoch 0: Validation Accuracy = 0.458
2026-02-22 20:43:00.535 | INFO     | modules.Train:run_training:147 - Epoch 0: Validation F1 = 0.44619509463728846
2026-02-22 20:43:02.215 | INFO     | modules.Train:run_training:88 - Epoch 1 :
2026-02-22 21:06:45.763 | INFO     | modules.Train:run_training:141 - Epoch 1: Train Loss = 0.6434338102887448
2026-02-22 21:06:45.764 | INFO     | modules.Train:run_tra

In [14]:
torch.load("./modules/train_savings/epoch_performances.pt")

{'epoch_train_losses': tensor([0.7147, 0.6434, 0.5957, 0.5627, 0.5291, 0.4931, 0.4598],
        dtype=torch.float64),
 'epoch_train_f1': tensor([0.5302, 0.6394, 0.6887, 0.7196, 0.7431, 0.7613, 0.7868],
        dtype=torch.float64),
 'epoch_train_accuracies': tensor([0.5261, 0.6331, 0.6838, 0.7152, 0.7393, 0.7578, 0.7840],
        dtype=torch.float64),
 'epoch_val_losses': tensor([0.7249, 0.7290, 0.6903, 0.7104, 0.7056, 0.9031, 0.7739],
        dtype=torch.float64),
 'epoch_val_f1': tensor([0.7249, 0.7290, 0.6903, 0.7104, 0.7056, 0.9031, 0.7739],
        dtype=torch.float64),
 'epoch_val_accuracies': tensor([0.4580, 0.5800, 0.6100, 0.5960, 0.6260, 0.5780, 0.6180],
        dtype=torch.float64)}